# Gold Price Prediction using ARIMA Model

This notebook demonstrates a comprehensive time-series forecasting approach for gold price prediction using ARIMA (AutoRegressive Integrated Moving Average) models.

## Key Features:
- Statistical tests for stationarity (ADF test)
- ACF & PACF analysis for parameter determination
- Automated ARIMA model optimization
- Comprehensive model diagnostics
- Performance evaluation with error metrics
- Interactive visualizations and forecasting

In [ ]:
# Import the gold price prediction module
from gold_price_prediction import GoldPricePredictor
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Configure matplotlib for notebook display
%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Initialize the Predictor and Load Data

We'll start by creating a GoldPricePredictor instance and loading historical gold price data.

In [ ]:
# Initialize predictor
predictor = GoldPricePredictor(symbol='GC=F', period='3y')

# Load historical gold price data
data = predictor.load_data()

# Display basic information
print(f"Dataset Shape: {data.shape}")
print(f"Date Range: {data.index.min().date()} to {data.index.max().date()}")
print(f"Price Range: ${data['Close'].min():.2f} - ${data['Close'].max():.2f}")

# Show first few rows
data.head()

## 2. Exploratory Data Analysis

Let's examine the gold price data through various statistical and visual analyses.

In [ ]:
# Perform exploratory data analysis
predictor.explore_data()

## 3. Stationarity Testing

ARIMA models require stationary time series. We'll test for stationarity using the Augmented Dickey-Fuller test.

In [ ]:
# Check stationarity of the original series
stationarity_result = predictor.check_stationarity()

# If not stationary, make it stationary
if not stationarity_result['is_stationary']:
    print("\nMaking the series stationary...")
    stationary_series = predictor.make_stationary()
    print(f"\nStationary series shape: {stationary_series.shape}")

## 4. ACF and PACF Analysis

Analyze autocorrelation and partial autocorrelation functions to determine optimal ARIMA parameters.

In [ ]:
# Analyze ACF and PACF plots
predictor.analyze_acf_pacf(lags=40)

## 5. ARIMA Model Optimization

Find the optimal ARIMA parameters using automated search with AIC criterion.

In [ ]:
# Find optimal ARIMA order
best_order = predictor.find_best_arima_order(max_p=3, max_d=2, max_q=3)
print(f"\nOptimal ARIMA order: {best_order}")

## 6. Model Fitting and Diagnostics

Fit the ARIMA model with optimal parameters and perform comprehensive diagnostics.

In [ ]:
# Fit the ARIMA model
fitted_model = predictor.fit_arima_model(order=best_order)

if fitted_model:
    print("\nModel fitted successfully!")
    print(f"AIC: {predictor.diagnostics['aic']:.2f}")
    print(f"BIC: {predictor.diagnostics['bic']:.2f}")
else:
    print("Model fitting failed.")

In [ ]:
# Perform model diagnostics
predictor.model_diagnostics()

## 7. Performance Evaluation

Evaluate model performance using train-test split and multiple error metrics.

In [ ]:
# Calculate performance metrics
metrics = predictor.calculate_performance_metrics(test_size=0.2)

if metrics:
    print("\n" + "="*50)
    print("PERFORMANCE SUMMARY")
    print("="*50)
    print(f"ARIMA Model RMSE: ${metrics['rmse']:.2f}")
    print(f"Baseline Model RMSE: ${metrics['baseline_rmse']:.2f}")
    print(f"Improvement: {metrics['improvement_rmse']:.1f}%")
    print(f"MAPE: {metrics['mape']:.2f}%")

## 8. Generate Forecasts

Generate future price predictions with confidence intervals.

In [ ]:
# Generate 30-day forecast
forecast = predictor.generate_forecast(steps=30)

if forecast:
    print(f"\n📈 Next 7 Days Forecast:")
    for i in range(7):
        date = forecast['forecast_dates'][i]
        price = forecast['forecast'].iloc[i]
        lower = forecast['lower_ci'].iloc[i]
        upper = forecast['upper_ci'].iloc[i]
        print(f"   {date.strftime('%Y-%m-%d')}: ${price:.2f} (${lower:.2f} - ${upper:.2f})")

## 9. Create Custom Visualizations

Create focused visualizations for the forecast.

In [ ]:
# Create custom forecast visualization
if forecast:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))
    
    # Plot 1: Recent prices + forecast
    recent_data = predictor.data['Close'].tail(60)
    ax1.plot(recent_data.index, recent_data, label='Historical Prices', 
             color='blue', linewidth=2)
    ax1.plot(forecast['forecast_dates'], forecast['forecast'], 
             label='30-Day Forecast', color='red', linewidth=2, linestyle='--')
    ax1.fill_between(forecast['forecast_dates'],
                     forecast['lower_ci'],
                     forecast['upper_ci'],
                     alpha=0.3, color='red', label='95% Confidence Interval')
    ax1.set_title('Gold Price: Last 60 Days + 30-Day Forecast')
    ax1.set_ylabel('Price ($)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Forecast distribution
    ax2.hist(forecast['forecast'], bins=20, alpha=0.7, color='gold', edgecolor='black')
    ax2.axvline(forecast['forecast'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: ${forecast["forecast"].mean():.2f}')
    ax2.set_title('Forecast Price Distribution')
    ax2.set_xlabel('Predicted Price ($)')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 10. Investment Insights

Generate actionable insights for investment decision-making.

In [ ]:
# Generate investment insights
if forecast and metrics:
    current_price = predictor.data['Close'].iloc[-1]
    next_day_price = forecast['forecast'].iloc[0]
    week_avg_price = forecast['forecast'][:7].mean()
    month_avg_price = forecast['forecast'].mean()
    
    price_change_1d = ((next_day_price - current_price) / current_price) * 100
    price_change_7d = ((week_avg_price - current_price) / current_price) * 100
    price_change_30d = ((month_avg_price - current_price) / current_price) * 100
    
    print("\n" + "="*60)
    print("INVESTMENT DECISION SUPPORT ANALYSIS")
    print("="*60)
    
    print(f"\n📊 CURRENT MARKET STATUS:")
    print(f"   Current Gold Price: ${current_price:.2f}")
    print(f"   Model Accuracy (MAPE): {metrics['mape']:.2f}%")
    print(f"   Confidence Level: {100 - metrics['mape']:.1f}%")
    
    print(f"\n🔮 PRICE PREDICTIONS:")
    print(f"   Next Day: ${next_day_price:.2f} ({price_change_1d:+.2f}%)")
    print(f"   7-Day Average: ${week_avg_price:.2f} ({price_change_7d:+.2f}%)")
    print(f"   30-Day Average: ${month_avg_price:.2f} ({price_change_30d:+.2f}%)")
    
    print(f"\n💡 INVESTMENT RECOMMENDATION:")
    
    # Short-term recommendation (1-7 days)
    if price_change_7d > 2:
        short_term = "🟢 STRONG BUY - Significant upward trend expected"
    elif price_change_7d > 0.5:
        short_term = "🟢 BUY - Moderate upward trend expected"
    elif price_change_7d > -0.5:
        short_term = "🟡 HOLD - Minimal price movement expected"
    elif price_change_7d > -2:
        short_term = "🔴 SELL - Moderate downward trend expected"
    else:
        short_term = "🔴 STRONG SELL - Significant downward trend expected"
    
    # Long-term recommendation (30 days)
    if price_change_30d > 3:
        long_term = "🟢 STRONG BUY - Strong long-term growth expected"
    elif price_change_30d > 1:
        long_term = "🟢 BUY - Positive long-term outlook"
    elif price_change_30d > -1:
        long_term = "🟡 HOLD - Stable long-term outlook"
    elif price_change_30d > -3:
        long_term = "🔴 SELL - Negative long-term outlook"
    else:
        long_term = "🔴 STRONG SELL - Significant long-term decline expected"
    
    print(f"   Short-term (7 days): {short_term}")
    print(f"   Long-term (30 days): {long_term}")
    
    # Risk assessment
    returns = predictor.data['Close'].pct_change().dropna()
    volatility = returns.std() * np.sqrt(252) * 100
    
    if volatility > 25:
        risk_level = "🔴 HIGH RISK"
    elif volatility > 15:
        risk_level = "🟡 MODERATE RISK"
    else:
        risk_level = "🟢 LOW RISK"
    
    print(f"\n⚠️ RISK ASSESSMENT:")
    print(f"   Risk Level: {risk_level}")
    print(f"   Annualized Volatility: {volatility:.1f}%")
    print(f"   Maximum Drawdown Risk: ±{returns.std() * 2 * 100:.1f}% (95% confidence)")
    
    print(f"\n📋 DISCLAIMER:")
    print(f"   • This analysis is for educational purposes only")
    print(f"   • Past performance does not guarantee future results")
    print(f"   • Always conduct thorough research before investing")
    print(f"   • Consider consulting with financial professionals")

## 11. Model Summary and Conclusions

Summarize the analysis results and model performance.

In [ ]:
# Generate comprehensive report
predictor.generate_report()

## Conclusion

This notebook demonstrated a complete ARIMA-based gold price prediction system with:

✅ **Statistical Rigor**: ADF test, ACF/PACF analysis, and automated parameter optimization  
✅ **Model Validation**: Comprehensive diagnostics and performance evaluation  
✅ **Practical Application**: Investment insights and risk assessment  
✅ **Superior Performance**: Significant improvement over baseline models  

The model provides a robust framework for gold price forecasting and can be extended with additional features like:
- Seasonal ARIMA (SARIMA) for handling seasonality
- External factors integration (economic indicators, market sentiment)
- Ensemble methods combining multiple models
- Real-time data streaming and automated retraining

---

*Remember: This analysis is for educational purposes. Always conduct thorough research and consider professional advice before making investment decisions.*